Lecture: AI I - Advanced 

Previous:
[**Chapter 2.3: Ensemble learning**](../03_ensemble.ipynb)

---

# Exercise 2.3: Ensemble learning

> Hint: When doing the exercises put your solution in the designated "Solution" section:
> ```python
> # Solution (put your code here)
> ```

## Task 1: Diabetes Regression with multiple Multi-Layer Perceptron

The diabetes dataset contains 442 samples with 10 baseline variables (age, sex, BMI, blood pressure, and 6 blood serum measurements). The target is a quantitative measure of disease progression one year after baseline.

**Tasks**
- Data Exploration & Understanding
- Data preparation
- Build Models: multiple Multi-Layer Perceptrons for regression (at least 3 with different architectures)
- Train the models 
- Construct an ensemble model from the individual models
- Evaluate the model performance using appropriate regression metrics (e.g. MSE, MAE) and Tensorboard


In [5]:
# prerequisites (don't edit this block)
from sklearn.datasets import load_diabetes

diabetes = load_diabetes()
x = diabetes.data
y = diabetes.target

In [6]:
# Solution (put your code here)

# === 1. Data Exploration & Understanding ===
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

torch.manual_seed(42)
np.random.seed(42)

# --- Data Exploration ---
print(f"Dataset shape:  x={x.shape}, y={y.shape}")
print(f"Features:       {diabetes.feature_names}")
print(f"Target range:   {y.min():.0f} – {y.max():.0f}  (mean={y.mean():.1f}, std={y.std():.1f})")

# === 2. Data Preparation ===
# StandardScaler für die Features (Target bleibt unverändert)
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)

# 60 / 20 / 20 Split  –– KEIN stratify, weil y kontinuierlich ist
x_train, x_temp, y_train, y_temp = train_test_split(
    x_scaled, y, test_size=0.4, random_state=42
)
x_val, x_test, y_val, y_test = train_test_split(
    x_temp, y_temp, test_size=0.5, random_state=42
)

# FloatTensor für Regression (LongTensor wäre nur für Klassifikation)
batch_size = 16
train_loader = DataLoader(
    TensorDataset(torch.FloatTensor(x_train), torch.FloatTensor(y_train)),
    batch_size=batch_size, shuffle=True,
)
val_loader = DataLoader(
    TensorDataset(torch.FloatTensor(x_val), torch.FloatTensor(y_val)),
    batch_size=batch_size,
)
test_loader = DataLoader(
    TensorDataset(torch.FloatTensor(x_test), torch.FloatTensor(y_test)),
    batch_size=batch_size,
)

print(f"\nSplit sizes:    train={len(x_train)}, val={len(x_val)}, test={len(x_test)}")


# ============================================================
# === 3. Build Models – drei verschiedene MLP-Architekturen ===
# ============================================================

class FunnelMLP(nn.Module):
    """Breit → Schmal  (128 → 64 → 32 → 1)"""
    def __init__(self, input_dim=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64),        nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32),         nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


class BrickMLP(nn.Module):
    """Gleichmäßig breit  (64 → 64 → 64 → 1)"""
    def __init__(self, input_dim=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 64),        nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 64),        nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


class DiamondMLP(nn.Module):
    """Schmal → Breit → Schmal  (32 → 128 → 32 → 1)"""
    def __init__(self, input_dim=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),  nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 128),        nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 32),        nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


# ========================
# === 4. Training Loop ===
# ========================

def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        preds = model(xb)
        loss = criterion(preds, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    for xb, yb in loader:
        preds = model(xb)
        total_loss += criterion(preds, yb).item() * xb.size(0)
    return total_loss / len(loader.dataset)


def train_model(name, model, train_loader, val_loader,
                num_epochs=300, lr=1e-3, patience=30):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    writer = SummaryWriter(log_dir=f"runs/ensemble/{name}_{timestamp}")

    best_val_loss = float("inf")
    patience_counter = 0

    for epoch in range(num_epochs):
        train_loss = train_epoch(model, train_loader, criterion, optimizer)
        val_loss = evaluate(model, val_loader, criterion)

        writer.add_scalars("Loss", {"Train": train_loss, "Val": val_loss}, epoch)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  {name}: Early stopping at epoch {epoch}")
                break

    writer.close()
    return best_val_loss


# --- Modelle instanziieren & trainieren ---
models = {
    "Funnel":  FunnelMLP(),
    "Brick":   BrickMLP(),
    "Diamond": DiamondMLP(),
}

print("Training individual models …")
for name, model in models.items():
    best = train_model(name, model, train_loader, val_loader)
    print(f"  {name:10s}  best val MSE = {best:.2f}")


# ===================================
# === 5. Ensemble (Mean Averaging) ===
# ===================================

@torch.no_grad()
def predict_ensemble(models, loader):
    """Durchschnitt der Vorhersagen aller Modelle (Soft Averaging)."""
    all_preds, all_targets = [], []
    for xb, yb in loader:
        preds = torch.stack([m(xb) for m in models]).mean(dim=0)
        all_preds.append(preds)
        all_targets.append(yb)
    return torch.cat(all_preds).numpy(), torch.cat(all_targets).numpy()


# =====================
# === 6. Evaluation ===
# =====================

print("\n" + "=" * 55)
print(f"{'Model':<16} {'MSE':>8} {'MAE':>8} {'R²':>8}")
print("-" * 55)

results = {}
for name, model in models.items():
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            preds.append(model(xb))
            targets.append(yb)
    p = torch.cat(preds).numpy()
    t = torch.cat(targets).numpy()
    mse = mean_squared_error(t, p)
    mae = mean_absolute_error(t, p)
    r2  = r2_score(t, p)
    results[name] = (mse, mae, r2)
    print(f"  {name:<14} {mse:8.2f} {mae:8.2f} {r2:8.4f}")

# Ensemble
ens_preds, ens_targets = predict_ensemble(models.values(), test_loader)
ens_mse = mean_squared_error(ens_targets, ens_preds)
ens_mae = mean_absolute_error(ens_targets, ens_preds)
ens_r2  = r2_score(ens_targets, ens_preds)
print("-" * 55)
print(f"  {'Ensemble':<14} {ens_mse:8.2f} {ens_mae:8.2f} {ens_r2:8.4f}")
print("=" * 55)

Dataset shape:  x=(442, 10), y=(442,)
Features:       ['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']
Target range:   25 – 346  (mean=152.1, std=77.0)

Split sizes:    train=265, val=88, test=89
Training individual models …
  Funnel: Early stopping at epoch 117
  Funnel      best val MSE = 2588.75
  Brick: Early stopping at epoch 166
  Brick       best val MSE = 2576.34
  Diamond: Early stopping at epoch 145
  Diamond     best val MSE = 2449.45

Model                 MSE      MAE       R²
-------------------------------------------------------
  Funnel          2969.04    44.70   0.4867
  Brick           3111.57    45.44   0.4621
  Diamond         3145.27    45.04   0.4563
-------------------------------------------------------
  Ensemble        3054.29    44.95   0.4720


In [ ]:
# Test case (don't edit this block)
assert True

---

Lecture: AI I - Advanced 

Next: [**Chapter 3.1: PyTorch Lightning**](../../03_advanced/01_lightning.ipynb)